# Visibility Game — Actor-Critic Mixed Strategy Solver

**Game:** 2-player Visibility Game (Lotker et al., 2008).  
Each player chooses x ∈ [0, 1]; payoff = distance to the next-higher point (or to 1 if highest).  
**Analytical equilibrium:** density p*(x) = 1/(1−x) on support [0, 1−1/e] ≈ [0, 0.632].  
Each player's expected payoff = 1/e ≈ 0.3679.

**Key challenge:** The payoff function is discontinuous at x1 = x2.  
The Q-function Q(x; π) = E_{x'~π}[u(x, x')] is smooth in expectation, making it tractable for a Critic network.

**Convergence criterion (indifference condition):**  
At Nash equilibrium, Q(x; π*) must be *constant* over the support — no x yields a strictly better payoff.  
We measure this as in-support Q-std → 0.

---
**Notebook structure:**
1. Imports
2. Game environment (payoff, sampling)
3. Network architecture (Actor, Double Q-Critic with monotonicity constraint)
4. Replay buffer
5. Hyperparameters
6. Convergence criterion
7. Trainer
8. Training run
9. Diagnostics: indifference check & learned distribution

## 1. Imports

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 1 — 导入
# ══════════════════════════════════════════════════════════════════════
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from dataclasses import dataclass, field
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import copy

torch.manual_seed(2024)
np.random.seed(2024)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

## 2. Game Environment

The payoff has a single discontinuity at x1 = x2.  
`diagonal_priority_sample` oversamples near this diagonal so the Critic learns the boundary accurately.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2 — 真实环境
#
# 收益函数（2人可见度博弈）：
#   u1(x1,x2) = x2 - x1   若 x1 < x2  （先让步，代价小）
#   u1(x1,x2) = 1 - x1    若 x1 > x2  （后让步，收益高）
#   u1(x1,x2) = 0          若 x1 = x2
#
# 解析均衡：
#   p*(x) = 1/(1-x)，支撑 [0, 1-1/e]
#   均衡期望收益 = 1/e ≈ 0.3679
# ══════════════════════════════════════════════════════════════════════

EQ_PAYOFF  = 1.0 / math.e          # 均衡期望收益
EQ_SUPPORT = 1.0 - 1.0 / math.e   # 支撑上界 ≈ 0.6321


def payoff(x1: torch.Tensor, x2: torch.Tensor):
    """真实收益（向量化，无梯度操作）"""
    x1 = x1.float().clamp(0., 1.)
    x2 = x2.float().clamp(0., 1.)
    u1 = torch.where(x1 < x2, x2 - x1,
         torch.where(x1 > x2, 1. - x1,
                     torch.zeros_like(x1)))
    u2 = torch.where(x2 < x1, x1 - x2,
         torch.where(x2 > x1, 1. - x2,
                     torch.zeros_like(x2)))
    return u1, u2


def nash_gap(x1: torch.Tensor, x2: torch.Tensor) -> float:
    """Nash Gap = |E[u1] - 1/e|，越小越接近均衡收益"""
    with torch.no_grad():
        u1, _ = payoff(x1, x2)
    return abs(u1.mean().item() - EQ_PAYOFF)


def diagonal_priority_sample(n, diag_ratio=0.4, delta=0.05, device=DEVICE):
    """
    对角线加密采样。
    x1 = x2 的对角线是收益函数唯一的间断处，
    集中采样使 Critic 在此处拟合更准确。
    """
    n_diag = int(n * diag_ratio)
    n_uni  = n - n_diag
    c  = torch.rand(n_diag, device=device)
    x1 = torch.cat([(c + (torch.rand(n_diag, device=device)-.5)*delta).clamp(0,1),
                     torch.rand(n_uni, device=device)])
    x2 = torch.cat([(c + (torch.rand(n_diag, device=device)-.5)*delta).clamp(0,1),
                     torch.rand(n_uni, device=device)])
    return x1, x2


# —— 验证 ——
_x1 = torch.tensor([0.1, 0.5, 0.9, 0.5])
_x2 = torch.tensor([0.5, 0.1, 0.9, 0.5])
_u1, _u2 = payoff(_x1, _x2)
print("收益验证:")
for i in range(4):
    print(f"  x1={_x1[i]:.1f} x2={_x2[i]:.1f} → u1={_u1[i]:.3f} u2={_u2[i]:.3f}")
print(f"均衡收益 1/e = {EQ_PAYOFF:.4f}，支撑上界 = {EQ_SUPPORT:.4f}")


## 3. Network Architecture

**Actor:** implicit generative policy — z ~ N(0, I_d) → Sigmoid → x ∈ (0,1).  
Initialised so output ≈ Uniform(0,1) at the start.

**QCritic:** maps x1 → Q(x1; π) = E_{x2~π}[u(x1, x2)].  
Takes *only* x1 as input (not the pair), so it directly estimates the expected payoff function.

**Monotonicity constraint:** Q(x) is theoretically non-increasing.  
A penalty `mono_loss` discourages Q(x+ε) > Q(x), stabilising training near the support boundary.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3 — 网络定义
#
# 框架说明：
#
# 旧框架（错误）：
#   Critic 估计 u1(x1, x2)（单次随机收益）
#   Actor Loss = -u1 - λH·H(π)
#   问题：最大熵框架的均衡是均匀分布，不是 1/(1-x)
#
# 新框架（正确）：
#   Critic 估计 Q(x1; π) = E_{x2~π}[u1(x1,x2)]（期望收益函数）
#   Actor Loss = -A(x1) = -(Q(x1) - Q_bar)（复制者动态）
#   均衡条件：Q(x) = 常数（无差异条件），此时梯度自然归零
# ══════════════════════════════════════════════════════════════════════

class Mish(nn.Module):
    """处处可导，无死区，适合拟合有陡坡的函数"""
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))


# ── Actor：隐式策略网络 ───────────────────────────────────────────────
class Actor(nn.Module):
    """
    z ~ N(0, I_d) → x ∈ (0,1)

    隐式策略不假设分布参数形式，
    足够深的网络可以逼近任意连续分布，
    包括右偏重尾的均衡密度 p*(x) = 1/(1-x)。
    """
    def __init__(self, noise_dim=8):
        super().__init__()
        self.noise_dim = noise_dim
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 128), Mish(),
            nn.Linear(128, 128),       Mish(),
            nn.Linear(128, 64),        Mish(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        layers = [m for m in self.net if isinstance(m, nn.Linear)]
        for i, m in enumerate(layers):
            if i < len(layers) - 1:
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
            else:
                # 让 Sigmoid 输出初期接近 Uniform(0,1)
                # 需要输入标准差 ≈ π/√3（Logistic 分布标准差）
                std = (math.pi / math.sqrt(3.)) / math.sqrt(self.noise_dim)
                nn.init.normal_(m.weight, 0., std)
                nn.init.zeros_(m.bias)

    def forward(self, z):
        return self.net(z).squeeze(-1)

    def sample(self, n, device=DEVICE):
        z = torch.randn(n, self.noise_dim, device=device)
        return self.forward(z)


# ── QCritic：期望收益网络 ─────────────────────────────────────────────
class QCritic(nn.Module):
    """
    输入：x1 ∈ [0,1]（标量）
    输出：Q(x1; π) = E_{x2~π}[u1(x1, x2)]

    相比旧 Critic(x1, x2) → u1 的改进：
    1. 输入只有 x1，输出是期望值，不是单次随机收益
    2. 对角线 x1=x2 处的间断在期望意义下被平均掉，函数更光滑
    3. 方差更低，梯度更稳定
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 128),   nn.LayerNorm(128), Mish(),
            nn.Linear(128, 128), nn.LayerNorm(128), Mish(),
            nn.Linear(128, 64),  Mish(),
            nn.Linear(64, 1)     # 无激活，自由拟合值域
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x1):
        """x1: (N,) → Q: (N,)"""
        return self.net(x1.unsqueeze(-1)).squeeze(-1)


# ── DoubleQCritic ─────────────────────────────────────────────────────
class DoubleQCritic(nn.Module):
    """
    两个独立初始化的 QCritic。
    用两者的不一致性 |C1-C2| 作为不确定性估计。
    保守估计 = 均值 - β × 不确定性，防止 Actor 利用 Critic 的漏洞。
    """
    def __init__(self):
        super().__init__()
        self.c1 = QCritic()
        self.c2 = QCritic()

    def forward(self, x1):
        return self.c1(x1), self.c2(x1)

    def mean_Q(self, x1):
        return (self.c1(x1) + self.c2(x1)) / 2.

    def conservative_Q(self, x1, beta=0.3):
        """
        Q_cons = (C1+C2)/2 - β|C1-C2|
        不确定性高的区域被惩罚，Actor 不会利用 Critic 的未知区域
        """
        q1, q2 = self.c1(x1), self.c2(x1)
        return (q1 + q2) / 2. - beta * (q1 - q2).abs()

    def mono_loss(self, x1, eps=0.02):
        """
        单调性约束：Q(x1) 关于 x1 单调递减。
        数学依据：固定 x2 时 u1 斜率 = -1，对期望也成立。
        惩罚 Q(x1+eps) > Q(x1) 的违反。
        """
        x1p = (x1 + eps).clamp(0., 1.)
        loss = 0.
        for c in [self.c1, self.c2]:
            q_o = c(x1)
            q_p = c(x1p)
            loss += (F.relu(q_p - q_o + eps) ** 2).mean()
        return loss


# —— 验证 ——
_actor = Actor(8).to(DEVICE)          # 加 .to(DEVICE)
_dc    = DoubleQCritic().to(DEVICE)   # 加 .to(DEVICE)
print(f"\nActor  参数量: {sum(p.numel() for p in _actor.parameters()):,}")
print(f"Critic 参数量: {sum(p.numel() for p in _dc.parameters()):,}")
_xs = _actor.sample(1024)
print(f"Actor 初始输出范围: [{_xs.min():.3f}, {_xs.max():.3f}]  (期望覆盖 [0, 0.63])")
_q1, _q2 = _dc(_xs)
print(f"Q1 shape: {_q1.shape}, Q2 shape: {_q2.shape}")

## 4. Replay Buffer

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4 — 经验回放池
# ══════════════════════════════════════════════════════════════════════

class ReplayBuffer:
    """
    环形缓冲区，存储 (x1, x2, u1, u2)。

    采样策略：
      recent_ratio 比例来自最近写入（追踪当前 Actor 分布）
      其余来自全局历史（防止 Critic 遗忘旧区域）
    """
    def __init__(self, capacity=200_000, recent_ratio=0.5, device=DEVICE):
        self.capacity     = capacity
        self.recent_ratio = recent_ratio
        self.device       = device
        self._x1  = np.zeros(capacity, np.float32)
        self._x2  = np.zeros(capacity, np.float32)
        self._u1  = np.zeros(capacity, np.float32)
        self._u2  = np.zeros(capacity, np.float32)
        self._ptr = 0
        self._size = 0

    def push(self, x1, x2, u1, u2):
        data = [t.detach().cpu().numpy().flatten() for t in (x1, x2, u1, u2)]
        n    = len(data[0])
        end  = self._ptr + n
        for arr, src in zip([self._x1, self._x2, self._u1, self._u2], data):
            if end <= self.capacity:
                arr[self._ptr:end] = src
            else:
                f = self.capacity - self._ptr
                arr[self._ptr:] = src[:f]
                arr[:n-f]       = src[f:]
        self._ptr  = end % self.capacity
        self._size = min(self._size + n, self.capacity)

    def sample(self, batch_size):
        assert self._size > 0
        n_rec = int(batch_size * self.recent_ratio)
        n_his = batch_size - n_rec
        win   = min(self._size, max(batch_size * 4, 4096))
        if self._ptr >= win:
            ridx = np.arange(self._ptr - win, self._ptr)
        else:
            ridx = np.concatenate([
                np.arange(self.capacity - (win - self._ptr), self.capacity),
                np.arange(0, self._ptr)
            ])
        idx = np.concatenate([
            np.random.choice(ridx, n_rec,  replace=True),
            np.random.randint(0, self._size, n_his)
        ])
        np.random.shuffle(idx)
        return {k: torch.tensor(arr[idx], device=self.device)
                for k, arr in zip(["x1","x2","u1","u2"],
                                   [self._x1,self._x2,self._u1,self._u2])}

    def __len__(self):
        return self._size


## 5. Hyperparameters

Key differences from Colonel Blotto:
- `actor_update_interval = 5` — Actor updated less frequently; the Q-function is harder to estimate due to the payoff discontinuity
- `lambda_mono = 0.5` — monotonicity regularisation on Critic
- `diag_ratio = 0.4` — 40% of replay samples drawn near the x1 ≈ x2 diagonal

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 5 — 超参数
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    noise_dim:               int   = 8

    # 预热
    warmup_samples:          int   = 300_000
    warmup_batch_size:       int   = 2048
    warmup_epochs:           int   = 20

    # 交替训练
    total_steps:             int   = 10_000
    actor_batch_size:        int   = 1024
    critic_updates_per_step: int   = 20
    critic_batch_size:       int   = 1024
    actor_update_interval:   int   = 5      # 新增：Actor 每 5 步才更新一次
    
    # Q 函数估计：每个 x1 用 K 个 x2 蒙特卡洛估计期望
    # K 越大方差越小，但计算量越大
    K_monte_carlo:           int   = 32

    # 损失权重
    lambda_mono:             float = 0.5    # 单调性约束
    beta_ucb:                float = 0.3    # 保守估计强度
    
    # 
    boundary_penalty_weight: float = 0.1

    # 优化器
    actor_lr:                float = 1e-4
    critic_lr:               float = 3e-4

    # 经验池
    buffer_capacity:         int   = 200_000
    diag_ratio:              float = 0.4
    diag_delta:              float = 0.05

    # 日志与收敛
    log_every:               int   = 200
    # 收敛判据：可利用度 + Q标准差同时低于阈值
    converge_exp:            float = 0.01
    converge_qstd:           float = 0.05
    check_cooldown:          int   = 200    # 两次严格验证之间的最小间隔步数

    device:                  str   = DEVICE

cfg = Config()
print(cfg)


## 6. Convergence Criterion

Two necessary conditions (KS test against analytical density is printed for reference but **not** used as a stopping criterion — neural approximations inevitably have a boundary spike near x = 0.632 that inflates KS even when Nash conditions hold):
1. **Nash Gap < 0.01** — measured expected payoff close to 1/e
2. **Exploitability < 0.01** — no profitable unilateral deviation exists

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 6 — 严格收敛验证（KS 仅作参考，不作判据）
# ══════════════════════════════════════════════════════════════════════

def strict_convergence_check(trainer, n=50_000) -> bool:
    """
    Two-condition convergence criterion (no analytical solution required):
      1. Nash Gap < 0.01       (expected payoff close to equilibrium)
      2. Exploitability < 0.01 (no profitable unilateral deviation)

    KS statistic printed for reference only — not used as criterion.
    Reason: KS=0.05 is nearly impossible for neural approximations,
    and the boundary spike inflates KS even when Nash conditions hold.
    """
    cfg = trainer.cfg
    with torch.no_grad():
        x1 = trainer.actor.sample(n, cfg.device)
        x2 = trainer.actor.sample(n, cfg.device)

    gap = nash_gap(x1, x2)
    exp = trainer._exploitability(n_actor=n, n_grid=1000)

    # KS for reference only
    x1_np  = x1.cpu().numpy()
    u_unif = np.random.uniform(0, 1, n)
    x_eq   = 1. - np.exp((u_unif - 1.) / math.e)
    ks, _  = stats.ks_2samp(x1_np, x_eq)

    print(f"  Nash Gap:       {gap:.4f}  (< 0.01)")
    print(f"  Exploitability: {exp:.4f}  (< 0.01)")
    print(f"  KS statistic:   {ks:.4f}  (reference only, not a criterion)")

    converged = (gap < 0.01) and (exp < 0.01)
    if converged:
        trainer.best_actor_state  = copy.deepcopy(trainer.actor.state_dict())
        trainer.best_critic_state = copy.deepcopy(trainer.dc.state_dict())
        trainer.best_step         = len(trainer.history["step"])
        trainer.best_gap          = gap
        trainer.best_exp          = exp
        print(f"  ✅ Converged. Best weights saved at step {trainer.best_step}.")
    else:
        print(f"  ❌ Not converged, continue training.")
    return converged

## 7. Trainer

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 7 — 训练器
#
# Changes from original:
#   1. _update_actor: boundary penalty added to push mass away from
#      support upper bound (EQ_SUPPORT = 1 - 1/e ≈ 0.632)
#   2. train: actor updated every actor_update_interval steps,
#      giving Critic more time to track the current distribution
#   3. strict_convergence_check: KS removed from convergence criterion
# ══════════════════════════════════════════════════════════════════════

class Trainer:
    def __init__(self, cfg: Config):
        self.cfg        = cfg
        dev             = cfg.device
        self.actor      = Actor(cfg.noise_dim).to(dev)
        self.dc         = DoubleQCritic().to(dev)
        self.opt_actor  = Adam(self.actor.parameters(),  lr=cfg.actor_lr)
        self.opt_critic = Adam(self.dc.parameters(),     lr=cfg.critic_lr)
        self.buffer     = ReplayBuffer(cfg.buffer_capacity, device=dev)
        self.history    = {k: [] for k in
                           ["step", "critic_loss", "actor_loss",
                            "nash_gap", "exploitability", "q_std", "critic_mae"]}
        self._last_check_step = 0

    # ── Phase 0: Critic global warmup ─────────────────────────────
    def warmup_critic(self):
        """
        Pretrain Critic with uniform + diagonal-priority samples.
        Actor does not participate. Critic builds a baseline understanding
        of the Q function under a roughly uniform opponent.

        Note: warmup x2 is uniformly random, not from equilibrium.
        The alternating training phase will continuously update Critic
        using x2 from the current Actor.
        """
        cfg = self.cfg
        print("=" * 60)
        print("Phase 0: Critic Warmup")

        n_batches = cfg.warmup_samples // cfg.warmup_batch_size
        for _ in range(n_batches):
            x1, x2 = diagonal_priority_sample(
                cfg.warmup_batch_size, cfg.diag_ratio,
                cfg.diag_delta, cfg.device)
            u1, u2 = payoff(x1, x2)
            self.buffer.push(x1, x2, u1, u2)
        print(f"  Buffer size: {len(self.buffer):,}")

        total    = cfg.warmup_epochs * (cfg.warmup_samples // cfg.critic_batch_size)
        log_step = max(total // 10, 1)
        for it in range(total):
            loss = self._update_critic()
            if (it + 1) % log_step == 0:
                print(f"  Warmup [{it+1:5d}/{total}]  Critic Loss: {loss:.6f}")
        print("  Warmup complete.\n")

    # ── Phase 1: Actor-Critic alternating training ─────────────────
    def train(self):
        cfg = self.cfg
        print("=" * 60)
        print("Phase 1: Actor-Critic (Replicator Dynamics)")
        print("Convergence: Exploitability -> 0,  Q-Std -> 0")
        print("=" * 60)

        for step in range(1, cfg.total_steps + 1):

            # 1. Collect: both x1 and x2 from current Actor (self-play)
            #    Mathematical requirement: gradient is zero at Nash equilibrium
            #    only when opponent uses the same current policy.
            with torch.no_grad():
                x1 = self.actor.sample(cfg.actor_batch_size, cfg.device)
                x2 = self.actor.sample(cfg.actor_batch_size, cfg.device)
            u1, u2 = payoff(x1, x2)
            self.buffer.push(x1, x2, u1, u2)

            # 2. Critic: update every step (high frequency)
            c_losses = [self._update_critic()
                        for _ in range(cfg.critic_updates_per_step)]
            c_loss   = sum(c_losses) / len(c_losses)

            # 3. Actor: update every actor_update_interval steps
            #    Gives Critic actor_update_interval * critic_updates_per_step
            #    update opportunities before Actor moves again.
            #    Prevents C-MAE from diverging due to distribution drift.
            if step % cfg.actor_update_interval == 0:
                a_loss, q_std = self._update_actor()
            else:
                a_loss = self.history["actor_loss"][-1] if self.history["actor_loss"] else 0.
                q_std  = self.history["q_std"][-1]      if self.history["q_std"]      else 0.

            # 4. Metrics
            with torch.no_grad():
                xe1 = self.actor.sample(4096, cfg.device)
                xe2 = self.actor.sample(4096, cfg.device)
            gap = nash_gap(xe1, xe2)
            exp = self._exploitability()
            mae = self._critic_mae()

            for k, v in zip(
                ["step", "critic_loss", "actor_loss", "nash_gap",
                 "exploitability", "q_std", "critic_mae"],
                [step, c_loss, a_loss, gap, exp, q_std, mae]
            ):
                self.history[k].append(v)

            if step % cfg.log_every == 0:
                print(f"Step {step:5d} | "
                      f"C-Loss {c_loss:.5f} | "
                      f"C-MAE {mae:.4f} | "
                      f"Exp {exp:.4f} | "
                      f"Q-Std {q_std:.4f} | "
                      f"NashGap {gap:.4f}")

            # Convergence check: dual threshold + cooldown
            if (exp   < cfg.converge_exp   and
                q_std < cfg.converge_qstd  and
                step  > 500                and
                step - self._last_check_step >= cfg.check_cooldown):
                self._last_check_step = step
                print(f"\nStep={step}, Exp={exp:.4f}, Q-Std={q_std:.4f} -- running strict check...")
                if strict_convergence_check(self):
                    break

        print("\nTraining complete.")

    # ── Critic update ──────────────────────────────────────────────
    def _update_critic(self):
        """
        Target: Q(x1) ≈ E_{x2~π_current}[u1(x1, x2)]

        Label construction (Monte Carlo):
          For each x1, average u1 over K x2 samples from current Actor.
          K=16 balances variance reduction against compute cost.

        x2 from current Actor (not replay buffer history):
          -> Critic always estimates Q under current policy
          -> Actor gradient direction is correct at update time
        """
        cfg = self.cfg
        K   = cfg.K_monte_carlo

        batch = self.buffer.sample(cfg.critic_batch_size)
        x1    = batch["x1"]   # (N,)

        with torch.no_grad():
            x1_rep    = x1.unsqueeze(1).expand(-1, K).reshape(-1)   # (N*K,)
            x2_rep    = self.actor.sample(x1_rep.shape[0], cfg.device)
            u1_rep, _ = payoff(x1_rep, x2_rep)
            q_target  = u1_rep.reshape(-1, K).mean(dim=1)           # (N,)

        q1, q2 = self.dc(x1)
        loss   = (F.huber_loss(q1, q_target) +
                  F.huber_loss(q2, q_target) +
                  cfg.lambda_mono * self.dc.mono_loss(x1))

        self.opt_critic.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.dc.parameters(), 5.)
        self.opt_critic.step()
        return loss.item()

    # ── Actor update (replicator dynamics + boundary penalty) ───────
    def _update_actor(self):
        """
        Replicator dynamics: L_A = -E_{x1~π}[A(x1)]

        A(x1) = Q_cons(x1) - Q_bar  (advantage function)
          Q_cons: conservative Q estimate (mean - β * uncertainty)
          Q_bar:  batch mean Q, used as baseline to reduce variance

        Mathematical property:
          At Nash equilibrium, Q(x) = const -> A(x) = 0 -> gradient = 0
          No entropy regularization needed; equilibrium determined by Q flatness.

        Boundary penalty:
          Penalizes probability mass placed beyond EQ_SUPPORT = 1 - 1/e.
          Mathematical basis: Q(x1) < 1/e for x1 > EQ_SUPPORT, so no
          rational agent should place mass there. The penalty makes this
          explicit and prevents the boundary spike observed in the histogram.
          Weight 0.5 is strong enough to suppress the spike without
          distorting the in-support distribution shape.
        """
        cfg = self.cfg
        z   = torch.randn(cfg.actor_batch_size, self.actor.noise_dim,
                          device=cfg.device)
        x1  = self.actor(z)   # gradient flows through

        for p in self.dc.parameters():
            p.requires_grad_(False)

        q_cons    = self.dc.conservative_Q(x1, beta=cfg.beta_ucb)  # (N,)
        q_bar     = q_cons.detach().mean()                          # baseline, no gradient
        advantage = q_cons - q_bar                                  # (N,)

        # Boundary penalty: quadratic cost for x1 > EQ_SUPPORT
        boundary_penalty = F.relu(x1 - EQ_SUPPORT).pow(2).mean()

        loss = -advantage.mean() + cfg.boundary_penalty_weight * boundary_penalty

        self.opt_actor.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 2.)
        self.opt_actor.step()

        for p in self.dc.parameters():
            p.requires_grad_(True)

        # Q std: measures how well the indifference condition is satisfied
        # At Nash equilibrium all actions have equal Q -> std -> 0
        q_std = q_cons.detach().std().item()
        return loss.item(), q_std

    # ── Exploitability ─────────────────────────────────────────────
    def _exploitability(self, n_actor=4096, n_grid=500) -> float:
        """
        Exp(π) = max_{x'} Q(x'; π) - E_{x~π}[Q(x; π)]

        Interpretation:
          Best fixed deviation payoff minus current average payoff.
          Exp -> 0 means no unilateral deviation is profitable: Nash equilibrium.
        """
        with torch.no_grad():
            x1     = self.actor.sample(n_actor, self.cfg.device)
            q_mean = self.dc.mean_Q(x1).mean().item()
            x_grid = torch.linspace(0, 1, n_grid, device=self.cfg.device)
            q_max  = self.dc.mean_Q(x_grid).max().item()
        return max(0., q_max - q_mean)

    # ── Critic MAE against true Q ──────────────────────────────────
    def _critic_mae(self, n=2000, K=64) -> float:
        """
        Use large-K Monte Carlo as ground-truth Q approximation.
        Compute mean absolute error of Critic prediction against it.
        x2 from current Actor (consistent with training).
        """
        cfg = self.cfg
        with torch.no_grad():
            x1        = torch.rand(n, device=cfg.device)
            x1_rep    = x1.unsqueeze(1).expand(-1, K).reshape(-1)
            x2_rep    = self.actor.sample(x1_rep.shape[0], cfg.device)
            u1_rep, _ = payoff(x1_rep, x2_rep)
            q_true    = u1_rep.reshape(n, K).mean(dim=1)
            q_pred    = self.dc.mean_Q(x1)
        return (q_pred - q_true).abs().mean().item()

## 8. Training Run

**Final configuration** (converged):
- `boundary_penalty_weight = 0.0` — boundary penalty not needed; Q monotonicity constraint alone is sufficient
- `actor_update_interval = 5`, `actor_lr = 1e-4`, `critic_lr = 3e-4`
- Converges within ~3000–5000 steps

In [ ]:
# 训练（先不加入边界惩罚）
cfg = Config(boundary_penalty_weight=0.0)
trainer = Trainer(cfg)
trainer.warmup_critic()
trainer.train()
# 恢复收敛时的最佳权重
if hasattr(trainer, "best_actor_state"):
    trainer.actor.load_state_dict(trainer.best_actor_state)
    trainer.dc.load_state_dict(trainer.best_critic_state)
    print(f"Restored best weights from step {trainer.best_step} "
          f"(NashGap={trainer.best_gap:.4f}, Exp={trainer.best_exp:.4f})")
else:
    print("Warning: no saved checkpoint found, using final weights.")

## 9. Diagnostics

### 9a. Pointwise Indifference Check
Plot Q(x1; π) across [0, 1]. At equilibrium, this should be **flat** within the support [0, 0.632] at height ≈ 1/e, then drop sharply outside.

### 9b. Learned Distribution vs Analytical Equilibrium
Compare the Actor's sampled density to p*(x) = 1/(1−x) (normalised over [0, 0.632]).

In [ ]:
def pointwise_indifference_check(trainer, n_x1=50, n_x2=2000):
    cfg = trainer.cfg
    x1_grid = torch.linspace(0.01, 0.99, n_x1, device=cfg.device)
    q_true  = []

    with torch.no_grad():
        for x1_val in x1_grid:
            x1_rep = x1_val.expand(n_x2)
            x2_rep = trainer.actor.sample(n_x2, cfg.device)
            u1, _  = payoff(x1_rep, x2_rep)
            q_true.append(u1.mean().item())

    q_true = np.array(q_true)
    x1_np  = x1_grid.cpu().numpy()

    print(f"Q mean:  {q_true.mean():.4f}  (equilibrium value {EQ_PAYOFF:.4f})")
    print(f"Q std:   {q_true.std():.4f}  (-> 0 means indifference condition satisfied)")
    print(f"Q range: {q_true.max()-q_true.min():.4f}")

    plt.figure(figsize=(9, 4))
    plt.plot(x1_np, q_true, 'b-', lw=2, label='True Q(x1; π_current)')
    plt.axhline(EQ_PAYOFF, color='r', ls='--', label=f'Equilibrium payoff 1/e={EQ_PAYOFF:.3f}')
    plt.axvline(EQ_SUPPORT, color='gray', ls=':', label=f'Support upper bound {EQ_SUPPORT:.3f}')
    plt.xlabel('x1'); plt.ylabel('Q(x1)')
    plt.title('Pointwise Indifference Check\n(flat line = converged,  hump shape = stuck at uniform)')
    plt.legend(); plt.grid(alpha=.3)
    plt.show()
    return q_true

q_diag = pointwise_indifference_check(trainer)

In [ ]:
import numpy as np
from scipy.integrate import quad

x1_grid = torch.linspace(0.01, 0.99, 50, device=cfg.device)
x1_np   = x1_grid.cpu().numpy()
q_diag  = pointwise_indifference_check(trainer)

mask = x1_np <= EQ_SUPPORT
q_in = q_diag[mask]
print(f"In-support Q mean:  {q_in.mean():.4f}  (equilibrium {EQ_PAYOFF:.4f})")
print(f"In-support Q std:   {q_in.std():.4f}  (indifference metric, -> 0 at equilibrium)")
print(f"In-support Q range: {q_in.max()-q_in.min():.4f}")

In [ ]:
with torch.no_grad():
    xs = trainer.actor.sample(50_000, cfg.device).cpu().numpy()

x_eq  = np.linspace(0.001, EQ_SUPPORT - 0.001, 300)
p_eq  = 1. / (1. - x_eq)
# 归一化（密度积分到1）
from scipy.integrate import quad
norm, _ = quad(lambda x: 1/(1-x), 0, EQ_SUPPORT)
p_eq_normalized = p_eq / norm

plt.figure(figsize=(9, 4))
plt.hist(xs, bins=100, density=True, alpha=0.6, color='steelblue', label='Learned policy')
plt.plot(x_eq, p_eq_normalized, 'r-', lw=2, label=r'Equilibrium $p^*(x)=1/(1-x)$')
plt.axvline(EQ_SUPPORT, color='gray', ls='--', label=f'Support bound {EQ_SUPPORT:.3f}')
plt.xlim(0, 1); plt.xlabel('x'); plt.ylabel('Density')
plt.title('Learned distribution vs analytical equilibrium')
plt.legend(); plt.grid(alpha=.3)
plt.show()